# CITD Image Processing — Colab/Kaggle Pipeline

This notebook is intentionally a thin orchestrator. It clones the repository, installs the declared dependencies, loads the Roboflow secret, prepares the dataset at runtime, trains YOLO through `scripts/train-yolo.py`, and runs the existing inference/evaluation commands. Kaggle and Colab use their active Python environment and preinstalled PyTorch; local runs use `uv`.

No model, dataset, or pipeline logic is duplicated here. The default ref is `develop`; override it with `LPR_REPO_REF` when testing an unmerged feature branch.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = os.getenv("LPR_REPO_URL", "https://github.com/cuongmn2011/CITD_ImageProcessing.git")
REPO_REF = os.getenv("LPR_REPO_REF", "develop")

if Path("/content").is_dir():
    WORK_ROOT = Path("/content")
elif Path("/kaggle/working").is_dir():
    WORK_ROOT = Path("/kaggle/working")
else:
    WORK_ROOT = Path.cwd()
REPO_DIR = WORK_ROOT / "CITD_ImageProcessing"

def run(command, *, cwd=REPO_DIR, env=None):
    print("$", " ".join(str(part) for part in command))
    completed = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        check=False,
        text=True,
    )
    if completed.returncode:
        raise subprocess.CalledProcessError(completed.returncode, command)
    return completed

if not (REPO_DIR / ".git").exists():
    run(["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(REPO_DIR)], cwd=WORK_ROOT)
else:
    run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR)
    run(["git", "checkout", REPO_REF], cwd=REPO_DIR)
    run(["git", "reset", "--hard", f"origin/{REPO_REF}"], cwd=REPO_DIR)

print("Repository:", REPO_DIR)
print("Revision:", REPO_REF)

In [ ]:
# Install the project's environment. Hosted Kaggle/Colab runtimes reuse their
# existing PyTorch instead of resolving the full uv lockfile.
import importlib.util

USE_SYSTEM_ENV = Path("/kaggle").is_dir() or Path("/content").is_dir()


def install_package(package_name, *, no_deps=False):
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--no-input",
        "--no-cache-dir",
        "-q",
    ]
    if no_deps:
        command.append("--no-deps")
    command.append(package_name)
    run(command, cwd=REPO_DIR)


def ensure_command(command_name, package_name):
    if shutil.which(command_name):
        print(f"{command_name}: already available")
        return
    install_package(package_name)


def ensure_python_package(module_name, package_name, *, no_deps=False):
    if importlib.util.find_spec(module_name) is not None:
        print(f"{module_name}: already available")
        return
    install_package(package_name, no_deps=no_deps)


ensure_python_package("ipywidgets", "ipywidgets")
if USE_SYSTEM_ENV:
    print("Hosted notebook detected; reusing preinstalled PyTorch")
    install_package(".", no_deps=True)
    ensure_python_package(
        "ultralytics",
        "ultralytics",
        no_deps=importlib.util.find_spec("torch") is not None,
    )
    ensure_python_package("ultralytics_thop", "ultralytics-thop", no_deps=True)
    ensure_python_package("roboflow", "roboflow")
    ensure_python_package("pytesseract", "pytesseract")
    UV = shutil.which("uv")
    LPR_COMMAND = [sys.executable, "-m", "lpr.cli"]
    PREPARE_DATASET_COMMAND = [sys.executable, "scripts/prepare-dataset.py"]
    TRAIN_COMMAND = [sys.executable, "scripts/train-yolo.py"]
else:
    ensure_command("uv", "uv")
    UV = shutil.which("uv")
    if UV is None:
        raise RuntimeError("uv executable was not found after installation")
    run([UV, "sync", "--extra", "vision", "--extra", "dataset", "--extra", "ocr"])
    LPR_COMMAND = [UV, "run", "lpr"]
    PREPARE_DATASET_COMMAND = [
        UV,
        "run",
        "--extra",
        "dataset",
        "python",
        "scripts/prepare-dataset.py",
    ]
    TRAIN_COMMAND = [
        UV,
        "run",
        "--extra",
        "vision",
        "--extra",
        "dataset",
        "python",
        "scripts/train-yolo.py",
    ]


In [ ]:
def load_roboflow_secret():
    value = os.getenv("ROBOFLOW_API_KEY")
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get("ROBOFLOW_API_KEY")
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
    except Exception:
        return None

ROBOFLOW_API_KEY = load_roboflow_secret()
if not ROBOFLOW_API_KEY:
    raise RuntimeError("Add ROBOFLOW_API_KEY to Colab userdata, Kaggle Secrets, or the environment")

ENV = os.environ.copy()
ENV["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY
ENV["MPLBACKEND"] = "Agg"
print("Roboflow secret loaded without printing its value.")

In [ ]:
DATASET_SPEC = os.getenv("LPR_ROBOFLOW_DATASET", "cuong-ta-ulxex/vietnamese-car-license-plate/1")
EPOCHS = int(os.getenv("LPR_EPOCHS", "50"))
IMAGE_SIZE = int(os.getenv("LPR_IMAGE_SIZE", "640"))
BATCH_SIZE = os.getenv("LPR_BATCH_SIZE", "-1")
DEVICE = os.getenv("LPR_DEVICE") or None
print({
    "dataset": DATASET_SPEC,
    "epochs": EPOCHS,
    "imgsz": IMAGE_SIZE,
    "batch": BATCH_SIZE,
    "device": DEVICE or "auto",
})

In [ ]:
# Download only when the local runtime cache is missing or invalid.
run([*PREPARE_DATASET_COMMAND, "--dataset", DATASET_SPEC], env=ENV)

In [ ]:
# Train with the repository's training script.
training_command = [
    *TRAIN_COMMAND,
    "--dataset", DATASET_SPEC,
    "--epochs", str(EPOCHS),
    "--imgsz", str(IMAGE_SIZE),
    "--batch", BATCH_SIZE,
]
if DEVICE is not None:
    training_command.extend(["--device", DEVICE])
run(training_command, env=ENV)

run_root = REPO_DIR / "runs/detect"
completed_runs = sorted(
    (
        path
        for path in run_root.glob("train*")
        if (path / "weights" / "best.pt").is_file()
    ),
    key=lambda path: path.stat().st_mtime,
)
if not completed_runs:
    raise FileNotFoundError("Training completed without producing weights/best.pt")
RUN_DIR = completed_runs[-1]
MODEL_PATH = RUN_DIR / "weights" / "best.pt"
print("Training run:", RUN_DIR)
print("Model:", MODEL_PATH)
import csv
import json
from IPython.display import FileLink, display

EXPORT_ROOT = (
    Path("/kaggle/working")
    if Path("/kaggle/working").is_dir()
    else WORK_ROOT / "outputs"
)
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR = EXPORT_ROOT / "citd-yolo11s-training"
if ARTIFACT_DIR.exists():
    shutil.rmtree(ARTIFACT_DIR)

shutil.copytree(RUN_DIR, ARTIFACT_DIR / "run")
shutil.copy2(MODEL_PATH, ARTIFACT_DIR / "best.pt")

last_model = RUN_DIR / "weights" / "last.pt"
if last_model.is_file():
    shutil.copy2(last_model, ARTIFACT_DIR / "last.pt")

metadata = {
    "dataset": DATASET_SPEC,
    "epochs": EPOCHS,
    "image_size": IMAGE_SIZE,
    "batch": BATCH_SIZE,
    "device": DEVICE or "auto",
    "training_run": str(RUN_DIR),
    "model": str(MODEL_PATH),
}
(ARTIFACT_DIR / "metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

ARCHIVE_PATH = Path(
    shutil.make_archive(
        str(EXPORT_ROOT / "citd-yolo11s-training"),
        "zip",
        root_dir=ARTIFACT_DIR,
    )
)
print("Training run:", RUN_DIR)
print("Best model:", MODEL_PATH)
print("Artifact directory:", ARTIFACT_DIR)
print("Archive:", ARCHIVE_PATH)
display(FileLink(str(ARCHIVE_PATH)))
display(FileLink(str(ARTIFACT_DIR / "best.pt")))

In [ ]:
# Show the final metrics and the best validation row.
results_csv = RUN_DIR / "results.csv"
if results_csv.is_file():
    with results_csv.open(newline="", encoding="utf-8") as file:
        metric_rows = list(csv.DictReader(file))
    metric_keys = (
        "epoch",
        "metrics/precision(B)",
        "metrics/recall(B)",
        "metrics/mAP50(B)",
        "metrics/mAP50-95(B)",
    )
    best_row = max(
        metric_rows,
        key=lambda row: float(row["metrics/mAP50-95(B)"]),
    )
    print("Best validation metrics:")
    print({key: best_row[key] for key in metric_keys})
    print("Final validation metrics:")
    print({key: metric_rows[-1][key] for key in metric_keys})
else:
    print(f"Metrics file not found: {results_csv}")

In [ ]:
# Tesseract is an OS package, not only a Python dependency.
if shutil.which("tesseract") is None:
    run(["apt-get", "update", "-qq"], cwd=REPO_DIR)
    run(["apt-get", "install", "-y", "-qq", "tesseract-ocr"], cwd=REPO_DIR)
print("Tesseract:", shutil.which("tesseract"))

In [ ]:
# Interactive video inference viewer.
import ipywidgets as widgets
from IPython.display import Video, clear_output, display

UI_OUTPUT_DIR = EXPORT_ROOT / "inference-output"
UI_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

video_input = widgets.Text(
    value=os.getenv("LPR_VIDEO_INPUT", ""),
    description="Video:",
    placeholder="/kaggle/input/.../video.mp4",
    layout=widgets.Layout(width="95%"),
)
video_output = widgets.Text(
    value=str(UI_OUTPUT_DIR / "annotated.mp4"),
    description="Output:",
    layout=widgets.Layout(width="95%"),
)
OCR_OPTIONS = ("tesseract",)
if importlib.util.find_spec("easyocr") is not None:
    OCR_OPTIONS += ("easyocr",)
else:
    print("EasyOCR is unavailable; the demo will use Tesseract.")
ocr_backend = widgets.Dropdown(
    options=OCR_OPTIONS,
    value="tesseract",
    description="OCR:",
)
device_input = widgets.Text(
    value=DEVICE or "auto",
    description="Device:",
)
max_frames = widgets.IntText(
    value=0,
    min=0,
    description="Max frames:",
)
run_video_button = widgets.Button(
    description="Run video inference",
    button_style="primary",
    icon="play",
)
video_output_area = widgets.Output()


def _run_video_inference(_button) -> None:
    with video_output_area:
        clear_output(wait=True)
        input_path = Path(video_input.value).expanduser()
        output_path = Path(video_output.value).expanduser()
        if not input_path.is_file():
            print(f"Video file not found: {input_path}")
            return
        output_path.parent.mkdir(parents=True, exist_ok=True)
        command = [
            *LPR_COMMAND,
            "infer-video",
            "--input",
            str(input_path),
            "--output",
            str(output_path),
            "--model",
            str(MODEL_PATH),
            "--ocr",
            ocr_backend.value,
        ]
        if device_input.value.strip() and device_input.value.strip().lower() != "auto":
            command.extend(["--device", device_input.value.strip()])
        if max_frames.value > 0:
            command.extend(["--max-frames", str(max_frames.value)])
        run(command, env=ENV)
        print(f"Annotated video: {output_path}")
        display(Video(str(output_path), embed=False, width=960))


run_video_button.on_click(_run_video_inference)
display(
    widgets.VBox(
        [
            video_input,
            video_output,
            widgets.HBox([ocr_backend, device_input, max_frames]),
            run_video_button,
            video_output_area,
        ]
    )
)

In [ ]:
# Interactive image inference preview and JSON result.
from IPython.display import Image as NotebookImage

image_input = widgets.Text(
    value=os.getenv("LPR_INPUT_IMAGE", ""),
    description="Image:",
    placeholder="/kaggle/input/.../car.jpg",
    layout=widgets.Layout(width="95%"),
)
run_image_button = widgets.Button(
    description="Run image inference",
    button_style="primary",
    icon="search",
)
image_output_area = widgets.Output()


def _run_image_inference(_button) -> None:
    with image_output_area:
        clear_output(wait=True)
        input_path = Path(image_input.value).expanduser()
        if not input_path.is_file():
            print(f"Image file not found: {input_path}")
            return
        display(NotebookImage(filename=str(input_path), width=960))
        command = [
            *LPR_COMMAND,
            "infer-image",
            "--image",
            str(input_path),
            "--model",
            str(MODEL_PATH),
            "--ocr",
            ocr_backend.value,
            "--variants",
            "otsu,clahe",
        ]
        if device_input.value.strip() and device_input.value.strip().lower() != "auto":
            command.extend(["--device", device_input.value.strip()])
        run(command, env=ENV)


run_image_button.on_click(_run_image_inference)
display(
    widgets.VBox(
        [
            image_input,
            run_image_button,
            image_output_area,
        ]
    )
)

In [ ]:
# Run the existing image pipeline on one validation image.
configured_image = os.getenv("LPR_INPUT_IMAGE")
if configured_image:
    sample_image = Path(configured_image).expanduser()
else:
    candidates = []
    for split in ("valid", "val", "test", "train"):
        candidates.extend(sorted((REPO_DIR / "data/processed/license-plates" / split / "images").glob("*")))
    sample_image = candidates[0] if candidates else None
if sample_image is None or not sample_image.is_file():
    raise FileNotFoundError("Set LPR_INPUT_IMAGE or provide a valid exported validation image")

run([
    *LPR_COMMAND,
    "infer-image",
    "--image", str(sample_image),
    "--model", str(MODEL_PATH),
    "--ocr", "tesseract",
    "--variants", "otsu,clahe",
], env=ENV)
print("Inference command completed; inspect the JSON output above.")

In [ ]:
# Optional OCR metrics. This evaluates OCR text, not detector mAP.
import csv

# Set this directly, or export LPR_OCR_CSV before running this cell:
# OCR_CSV_PATH = "/kaggle/input/<dataset-name>/ocr-ground-truth.csv"
OCR_CSV_PATH = os.getenv("LPR_OCR_CSV", "")

if not OCR_CSV_PATH and Path("/kaggle/input").is_dir():
    csv_candidates = []
    for candidate in sorted(Path("/kaggle/input").glob("**/*.csv")):
        try:
            with candidate.open(newline="", encoding="utf-8") as file:
                columns = set(csv.DictReader(file).fieldnames or [])
        except (OSError, UnicodeDecodeError):
            continue
        if {"ground_truth", "prediction"} <= columns:
            csv_candidates.append(candidate)

    if len(csv_candidates) == 1:
        OCR_CSV_PATH = str(csv_candidates[0])
        print(f"Auto-selected OCR CSV: {OCR_CSV_PATH}")
    elif csv_candidates:
        print("Multiple OCR CSV files found; set OCR_CSV_PATH explicitly:")
        for candidate in csv_candidates:
            print(f"  {candidate}")

if not OCR_CSV_PATH:
    print(
        "OCR evaluation skipped. Set OCR_CSV_PATH or LPR_OCR_CSV "
        "to a CSV containing ground_truth,prediction columns."
    )
else:
    ocr_csv_path = Path(OCR_CSV_PATH).expanduser()
    if not ocr_csv_path.is_file():
        print(f"OCR evaluation skipped; file not found: {ocr_csv_path}")
    else:
        run([*LPR_COMMAND, "evaluate-ocr", "--csv", str(ocr_csv_path)], env=ENV)

## Reproducibility checklist

Record `LPR_REPO_REF`, `DATASET_SPEC`, epoch/image-size/batch/device settings, generated model path, and the output metrics in the final report. Do not commit runtime secrets, downloaded data, or generated model weights.